# Load dataset 

In [5]:
from datasets import load_dataset

dataset = load_dataset("dair-ai/emotion")
train_data = dataset["train"]
test_data = dataset["test"]

In [6]:
from collections import defaultdict
from datasets import Dataset

few_shot = []
counts = defaultdict(int)

for ex in train_data:
    label = ex["label"]

    if counts[label] < 5:
        few_shot.append(ex)
        counts[label] += 1

    if sum(counts.values()) == 30:  # 6 classes × 5 examples
        break

few_shot_train = Dataset.from_list(few_shot)

print("Few-shot samples:", len(few_shot_train))

Few-shot samples: 30


In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True
    )

train_tok = few_shot_train.map(tokenize)
test_tok = dataset["test"].select(range(500)).map(tokenize)

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [9]:
train_tok.set_format("torch")
test_tok.set_format("torch")

In [11]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./fewshot_emotion",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,   # few-shot usually needs more epochs
    eval_strategy="epoch",
    logging_steps=5,
    save_strategy="no"
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,1.772470
2,1.777047,1.769485
3,1.691011,1.766346
4,1.544859,1.747658
5,1.353678,1.738509
6,1.353678,1.739420
7,1.211644,1.720481
8,1.090639,1.706609
9,0.966189,1.710908
10,0.910860,1.715738


TrainOutput(global_step=40, training_loss=1.3182408809661865, metrics={'train_runtime': 18.0702, 'train_samples_per_second': 16.602, 'train_steps_per_second': 2.214, 'total_flos': 39743054438400.0, 'train_loss': 1.3182408809661865, 'epoch': 10.0})

# Inference 

In [15]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

text = "I feel extremely happy and excited!"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

# MOVE INPUTS TO SAME DEVICE
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

pred = torch.argmax(outputs.logits, dim=1).item()

print("Predicted label:", pred)

Predicted label: 5


In [16]:
import json
import numpy as np

# =========================
# 1. FEW-SHOT PROMPT SETUP
# =========================

FEW_SHOT_PROMPT = """
You are an autonomous driving scene analyzer.

Return output strictly in JSON:
{"car": int, "truck": int, "bus": int, "pedestrian": int, "bicycle": int, "motorcycle": int}

---

Example 1:
Input: A street with two cars and one pedestrian crossing.
Output: {"car": 2, "truck": 0, "bus": 0, "pedestrian": 1, "bicycle": 0, "motorcycle": 0}

Example 2:
Input: Highway with 3 trucks and 1 motorcycle.
Output: {"car": 0, "truck": 3, "bus": 0, "pedestrian": 0, "bicycle": 0, "motorcycle": 1}

Example 3:
Input: Intersection with buses, cars, and cyclists.
Output: {"car": 5, "truck": 0, "bus": 2, "pedestrian": 1, "bicycle": 3, "motorcycle": 0}

---

Now solve:
Input: 
"""


# =========================
# 2. MOCK MODEL CALL
# =========================
# Replace this with GPT / LLM API call

def mock_llm(prompt):
    """
    Fake model for demo.
    Replace with OpenAI / local LLM.
    """
    # simple heuristic fallback (for testing only)
    return json.dumps({
        "car": 1,
        "truck": 0,
        "bus": 0,
        "pedestrian": 1,
        "bicycle": 0,
        "motorcycle": 0
    })


# =========================
# 3. LOAD DATASET (1000 samples)
# =========================

def load_dataset():
    """
    Replace this with real DAIR-AI dataset loader.
    Format:
    [
        {"input": "...", "label": {...}},
        ...
    ]
    """
    data = []
    for i in range(1000):
        data.append({
            "input": "A street with cars and pedestrians " + str(i),
            "label": {
                "car": 2,
                "truck": 0,
                "bus": 0,
                "pedestrian": 1,
                "bicycle": 0,
                "motorcycle": 0
            }
        })
    return data


# =========================
# 4. RUN INFERENCE
# =========================

def run_inference(dataset):
    results = []

    for sample in dataset:
        prompt = FEW_SHOT_PROMPT + sample["input"]

        output = mock_llm(prompt)
        pred = json.loads(output)

        results.append({
            "pred": pred,
            "gt": sample["label"]
        })

    return results


# =========================
# 5. METRICS
# =========================

def compute_metrics(results):
    classes = ["car", "truck", "bus", "pedestrian", "bicycle", "motorcycle"]

    mae_per_class = {c: [] for c in classes}
    rmse_per_class = {c: [] for c in classes}

    for r in results:
        for c in classes:
            p = r["pred"][c]
            g = r["gt"][c]

            err = abs(p - g)

            mae_per_class[c].append(err)
            rmse_per_class[c].append(err ** 2)

    print("\n===== METRICS =====")

    for c in classes:
        mae = np.mean(mae_per_class[c])
        rmse = np.sqrt(np.mean(rmse_per_class[c]))

        print(f"{c}: MAE={mae:.3f}, RMSE={rmse:.3f}")

    # overall
    all_mae = np.mean([np.mean(mae_per_class[c]) for c in classes])
    print("\nOVERALL MAE:", all_mae)


# =========================
# 6. MAIN PIPELINE
# =========================

if __name__ == "__main__":
    dataset = load_dataset()          # 1000 samples
    results = run_inference(dataset)  # few-shot inference
    compute_metrics(results)


===== METRICS =====
car: MAE=1.000, RMSE=1.000
truck: MAE=0.000, RMSE=0.000
bus: MAE=0.000, RMSE=0.000
pedestrian: MAE=0.000, RMSE=0.000
bicycle: MAE=0.000, RMSE=0.000
motorcycle: MAE=0.000, RMSE=0.000

OVERALL MAE: 0.16666666666666666
